<a href="https://colab.research.google.com/github/cassiecinzori/ECON3916/blob/main/Labs/Lecture24/ECON3916_Lab24.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 24: Causal ML - Double Machine Learning
## ECON 3916: Data Science for Economists
### Guided Construction Lab | 35 min Core + 10 min Extension

---

**Learning Objectives:**
- Demonstrate the regularization bias problem when LASSO is used for causal estimation
- Set up a DoubleML data object with correct outcome, treatment, and covariate columns
- Fit a Partially Linear Regression (PLR) model using Random Forest nuisance learners
- Interpret the Average Treatment Effect (ATE) and its confidence interval from DML output
- Estimate Conditional Average Treatment Effects (CATE) by income subgroup
- Assess robustness of causal estimates with sensitivity analysis

**Dataset:** 401(k) pension plan participation data (Chernozhukov & Hansen) - `doubleml.datasets.fetch_401K`

**Time estimate:** ~45 minutes

**Foundations First Policy:** Parts 1-3 and 5 are GUIDED (run as-is, interpret results). Parts 4 and 6 have YOUR TASK sections. Part 7 is the AI expansion.

---

In [ ]:
# -----------------------------------------------------------
# GUIDED - Run as-is
# Install required packages (Colab-safe)
# -----------------------------------------------------------
!pip install -q doubleml econml

In [ ]:
# -----------------------------------------------------------
# GUIDED - Run as-is
# Import libraries and load 401(k) data
# -----------------------------------------------------------
from doubleml import DoubleMLData, DoubleMLPLR
from doubleml.datasets import fetch_401K
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LassoCV
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# Load 401(k) data
data = fetch_401K(return_type='DataFrame')

print(f'Dataset shape: {data.shape}')
print(f'Columns: {list(data.columns)}')
print()
print('Key variables:')
print('  net_tfa  - Net total financial assets (outcome Y)')
print('  e401     - 401(k) eligibility indicator (treatment D)')
print('  Others   - Age, income, family size, education, etc. (covariates X)')
print()
data.head()

In [ ]:
# -----------------------------------------------------------
# GUIDED - Run as-is
# EDA: Summary statistics and treatment balance check
# -----------------------------------------------------------

print('=== Summary Statistics ===')
print(data[['net_tfa', 'e401', 'inc', 'age', 'fsize', 'educ']].describe().round(2))
print()

# Treatment balance check
treated = data[data['e401'] == 1]
control = data[data['e401'] == 0]

print('=== Treatment Balance Check ===')
print(f'Treated (401k eligible):     {len(treated):,} ({len(treated)/len(data)*100:.1f}%)')
print(f'Control (not eligible):      {len(control):,} ({len(control)/len(data)*100:.1f}%)')
print()

balance_vars = ['inc', 'age', 'fsize', 'educ', 'marr']
print(f'{"Variable":>10s}  {"Treated":>10s}  {"Control":>10s}  {"Diff":>10s}')
print('-' * 45)
for v in balance_vars:
    t_mean = treated[v].mean()
    c_mean = control[v].mean()
    diff = t_mean - c_mean
    print(f'{v:>10s}  {t_mean:10.2f}  {c_mean:10.2f}  {diff:+10.2f}')

print()
print('NOTE: The treated group has higher income and education.')
print('This is selection bias - firms offering 401(k) tend to have higher-paid workers.')
print('A naive comparison of outcomes will OVERESTIMATE the 401(k) effect.')

## Part 1: The Regularization Bias Problem (GUIDED)

Before we use DML, let us see **why** you cannot just throw LASSO at a causal
question. When LASSO penalizes the treatment coefficient, it shrinks the causal
effect toward zero - producing **regularization bias**.

We demonstrate on simulated data where we KNOW the true effect is exactly 5.0.

**The key insight:** ML models optimize prediction, not causal identification.
LASSO treats the treatment variable $D$ the same as any other covariate $X$ -
it has no way to know that $D$ is the variable whose coefficient we care about.

In [ ]:
# -----------------------------------------------------------
# GUIDED - Run as-is
# Demonstrate naive LASSO bias on simulated DGP
# TRUE ATE = 5.0
# -----------------------------------------------------------

# Simulate data with known causal effect
np.random.seed(42)
n = 5000
p = 100  # high-dimensional covariates

TRUE_ATE = 5.0

# Generate covariates
X_sim = np.random.normal(0, 1, size=(n, p))

# Treatment depends on some covariates (confounding)
propensity = 1 / (1 + np.exp(-(0.5 * X_sim[:, 0] + 0.3 * X_sim[:, 1] + 0.2 * X_sim[:, 2])))
D_sim = np.random.binomial(1, propensity)

# Outcome depends on treatment + covariates + noise
Y_sim = (TRUE_ATE * D_sim
         + 2.0 * X_sim[:, 0] + 1.5 * X_sim[:, 1] + 1.0 * X_sim[:, 2]
         + 0.5 * X_sim[:, 3] + 0.3 * X_sim[:, 4]
         + np.random.normal(0, 1, n))

# Naive approach: Run LASSO on Y ~ D + X
from sklearn.linear_model import LassoCV

X_with_D = np.column_stack([D_sim, X_sim])
lasso = LassoCV(cv=5, random_state=42, max_iter=10000)
lasso.fit(X_with_D, Y_sim)

lasso_ate = lasso.coef_[0]  # coefficient on D (first column)

print('=== Naive LASSO Approach ===')
print(f'True ATE:      {TRUE_ATE:.2f}')
print(f'LASSO ATE:     {lasso_ate:.2f}')
print(f'Bias:          {lasso_ate - TRUE_ATE:+.2f}')
print(f'Bias (%):      {(lasso_ate - TRUE_ATE) / TRUE_ATE * 100:+.1f}%')
print()
print('LASSO shrank the treatment coefficient toward zero.')
print('This is REGULARIZATION BIAS - the penalty does not distinguish')
print('the causal variable D from the nuisance covariates X.')
print()
print('Non-zero LASSO coefficients:', np.sum(lasso.coef_ != 0), 'out of', len(lasso.coef_))

In [ ]:
# -----------------------------------------------------------
# YOUR TASK -- Set up DoubleMLData for the 401(k) data
# -----------------------------------------------------------

y_col = 'net_tfa'       # Outcome: net total financial assets
d_cols = 'e401'         # Treatment: 401(k) eligibility indicator
x_cols = [c for c in data.columns if c not in ['net_tfa', 'e401']]  # All other columns are covariates

dml_data = DoubleMLData(
    data,
    y_col=y_col,
    d_cols=d_cols,
    x_cols=x_cols
)

print(dml_data)
print(f'\nOutcome variable: {y_col}')
print(f'Treatment variable: {d_cols}')
print(f'Covariates ({len(x_cols)}): {x_cols}')


**Interpretation (DoubleMLData Setup):**

The `DoubleMLData` object organizes the dataset into the three roles the PLR model requires: outcome `y` (net financial assets), treatment `d` (401(k) eligibility), and covariates `x` (everything else). Separating these roles explicitly is what distinguishes DML from a naive regression - the framework treats the treatment variable as the object of causal interest and all other variables as nuisance, whereas LASSO or OLS have no such distinction. The covariate set includes income, age, family size, education, and marital status - the variables most likely to drive both which workers are offered 401(k) plans and how much they save - which is precisely the confounding structure DML is designed to handle.


In [ ]:
# -----------------------------------------------------------
# YOUR TASK -- Define nuisance learners and fit DML
# -----------------------------------------------------------

# Two Random Forest nuisance models:
#   ml_l: predicts Y from X (outcome model)
#   ml_m: predicts D from X (treatment/propensity model)
ml_l = RandomForestRegressor(
    n_estimators=200,   # 200 trees for stable predictions
    max_depth=5,        # Shallow trees to avoid overfitting
    random_state=42
)

ml_m = RandomForestRegressor(
    n_estimators=200,
    max_depth=5,
    random_state=42
)

# Partially Linear Regression with 5-fold cross-fitting
dml_plr = DoubleMLPLR(
    dml_data,
    ml_l=ml_l,
    ml_m=ml_m,
    n_folds=5,   # 5-fold cross-fitting ensures nuisance models are fit out-of-sample
    n_rep=1
)

print('Fitting DML model with 5-fold cross-fitting...')
print('(This may take 30-60 seconds)')
dml_plr.fit()

print('\nDML model fitted successfully.')


**Interpretation (DML Model Setup):**

Random Forests are chosen as nuisance learners because they flexibly capture nonlinear relationships between covariates and both the outcome and treatment without requiring manual feature engineering. `max_depth=5` constrains tree complexity to reduce overfitting - the goal of the nuisance models is accurate out-of-sample prediction of residuals, not fitting the training data perfectly. The 5-fold cross-fitting is the key innovation of DML: each observation's residuals are computed using a nuisance model trained on the other four folds, which prevents the regularization bias that occurs when the same data is used to fit nuisance models and estimate the treatment effect. This sample-splitting is what allows DML to use flexible ML models while still producing valid, asymptotically normal inference on the causal parameter.


In [ ]:
# -----------------------------------------------------------
# GUIDED - Run as-is
# Print DML summary and interpret ATE
# -----------------------------------------------------------

print(dml_plr.summary)
print()

# Extract key results
ate = dml_plr.coef[0]
se = dml_plr.se[0]
ci = dml_plr.confint(level=0.95)

print('=== DML Results: Effect of 401(k) Eligibility on Net Financial Assets ===')
print(f'Average Treatment Effect (ATE): ${ate:,.0f}')
print(f'Standard Error:                 ${se:,.0f}')
print(f'95% Confidence Interval:        [${ci.iloc[0, 0]:,.0f}, ${ci.iloc[0, 1]:,.0f}]')
print(f't-statistic:                    {dml_plr.t_stat[0]:.2f}')
print(f'p-value:                        {dml_plr.pval[0]:.6f}')
print()
print('Interpretation:')
print(f'  Being eligible for a 401(k) plan increases net financial assets')
print(f'  by approximately ${ate:,.0f}, controlling for income, age, education,')
print(f'  family size, and other covariates.')
print()
if dml_plr.pval[0] < 0.05:
    print('  The effect is statistically significant at the 5% level.')
else:
    print('  The effect is NOT statistically significant at the 5% level.')

In [ ]:
# -----------------------------------------------------------
# YOUR TASK -- Estimate ATE by income quartile (CATE analysis)
# -----------------------------------------------------------

# Step 1: Create income quartiles
data['inc_quartile'] = pd.qcut(data['inc'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])

print('Income quartile ranges:')
for q in ['Q1', 'Q2', 'Q3', 'Q4']:
    subset = data[data['inc_quartile'] == q]
    print(f'  {q}: ${subset["inc"].min():,.0f} - ${subset["inc"].max():,.0f} '
          f'(n={len(subset):,})')
print()

# Step 2: Estimate DML separately for each income quartile
cate_results = []

for q in ['Q1', 'Q2', 'Q3', 'Q4']:
    print(f'Fitting DML for {q}...')
    subset = data[data['inc_quartile'] == q].drop(columns=['inc_quartile'])

    # Build DoubleMLData for this subgroup
    x_cols_q = [c for c in subset.columns if c not in ['net_tfa', 'e401']]
    dml_data_q = DoubleMLData(
        subset,
        y_col='net_tfa',
        d_cols='e401',
        x_cols=x_cols_q
    )

    ml_l_q = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42)
    ml_m_q = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42)

    dml_q = DoubleMLPLR(
        dml_data_q,
        ml_l=ml_l_q,
        ml_m=ml_m_q,
        n_folds=5,
        n_rep=1
    )
    dml_q.fit()

    ci = dml_q.confint(level=0.95)
    cate_results.append({
        'quartile': q,
        'ate': dml_q.coef[0],
        'se': dml_q.se[0],
        'ci_lower': ci.iloc[0, 0],
        'ci_upper': ci.iloc[0, 1]
    })

# Print results table
cate_df = pd.DataFrame(cate_results)
print()
print('=== CATE by Income Quartile ===')
for _, row in cate_df.iterrows():
    print(f'{row["quartile"]}: ATE = ${row["ate"]:>10,.0f}  '
          f'95% CI = [${row["ci_lower"]:>10,.0f}, ${row["ci_upper"]:>10,.0f}]')


**Interpretation (CATE by Income Quartile):**

Estimating DML separately within each income quartile allows the treatment effect to vary freely across subgroups - this is a subgroup CATE analysis rather than a single pooled ATE. The results typically show a gradient: higher-income households exhibit a larger dollar-denominated treatment effect from 401(k) eligibility, which is economically intuitive because they have more disposable income to redirect into retirement accounts and face higher marginal tax rates that make tax-deferred savings more valuable. Lower-income quartiles may show smaller or noisier effects, reflecting liquidity constraints that limit how much an eligibility offer can translate into actual asset accumulation, wider confidence intervals are expected in subgroup analyses due to smaller effective sample sizes, and any quartile whose CI crosses zero suggests the effect is not statistically distinguishable from zero for that income band.


In [ ]:
# -----------------------------------------------------------
# GUIDED - Run as-is
# Plot CATE by income quartile (bar chart with error bars)
# -----------------------------------------------------------

if len(cate_results) > 0:
    cate_df = pd.DataFrame(cate_results)

    fig, ax = plt.subplots(figsize=(8, 5))

    x_pos = range(len(cate_df))
    bars = ax.bar(x_pos, cate_df['ate'],
                  yerr=[cate_df['ate'] - cate_df['ci_lower'],
                        cate_df['ci_upper'] - cate_df['ate']],
                  capsize=8, color=['#3498db', '#2ecc71', '#e67e22', '#e74c3c'],
                  edgecolor='white', linewidth=1.5, alpha=0.85)

    ax.set_xticks(x_pos)
    ax.set_xticklabels(cate_df['quartile'])
    ax.set_xlabel('Income Quartile', fontsize=12)
    ax.set_ylabel('Estimated ATE ($)', fontsize=12)
    ax.set_title('Conditional ATE of 401(k) Eligibility by Income Quartile',
                 fontsize=13)
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)

    # Add value labels on bars
    for i, (_, row) in enumerate(cate_df.iterrows()):
        ax.text(i, row['ate'] + (row['ci_upper'] - row['ate']) + 200,
                f'${row["ate"]:,.0f}', ha='center', va='bottom', fontsize=10,
                fontweight='bold')

    plt.tight_layout()
    plt.show()

    print('Error bars show 95% confidence intervals.')
    print('Wider bars = more uncertainty in that subgroup estimate.')
else:
    print('Complete the CATE task above first, then re-run this cell.')

## Part 6: Interpretation Questions

Answer these questions in your own words (2-3 sentences each):

**Q1:** Why does the 401(k) effect vary across income quartiles? What economic mechanism might explain a larger effect for higher-income households?

*Answer:* Higher-income households face higher marginal tax rates, making the tax-deferred nature of 401(k) contributions more financially valuable to them. They also have more discretionary income available to redirect into retirement savings when an employer offers a plan, whereas lower-income households may be liquidity-constrained and unable to contribute meaningfully even when eligible. Additionally, higher earners are more likely to have existing financial knowledge and employer match programs that amplify the effect of eligibility.

**Q2:** If a policymaker wanted to maximize the impact of a retirement savings program, which income group should they target? What are the trade-offs between efficiency (largest effect) and equity (helping those who need it most)?

*Answer:* Purely on efficiency grounds, targeting upper-income quartiles would maximize the dollar increase in financial assets per dollar of policy effort, since the CATE estimates are largest there. However, equity considerations point in the opposite direction: lower-income households have the least retirement security and the greatest need for savings support, even if their behavioral response to eligibility is smaller. A policy optimizing both dimensions might focus on structural interventions for low-income workers - automatic enrollment, employer match mandates, or direct subsidies - that reduce the liquidity constraints limiting their treatment response.

**Q3:** Why did LASSO produce a biased estimate of the treatment effect in Part 1, but DML did not? What is the fundamental difference in how DML handles the treatment variable?

*Answer:* LASSO treats the treatment variable D identically to every other covariate - its penalty shrinks all coefficients toward zero, including the one on D, producing regularization bias in the causal estimate. DML solves this by partialling out the effect of covariates X separately from both Y and D using cross-fitted nuisance models, then estimating the treatment effect from the residual-on-residual regression where no further penalization is applied to the treatment coefficient. The cross-fitting step ensures the nuisance predictions are computed out-of-sample, which is what allows DML to use flexible ML learners without introducing the regularization bias that contaminates the naive LASSO approach.


In [ ]:
# -----------------------------------------------------------
# AI EXPANSION - P.R.I.M.E. Prompt
# -----------------------------------------------------------

# Copy and paste this prompt into Claude or ChatGPT:
#
# """Use the DoubleML sensitivity_analysis() method to assess how
# robust the ATE is to potential unmeasured confounders.
#
# Specifically:
# 1. Run dml_plr.sensitivity_analysis() with cf_y=0.03, cf_d=0.03
#    (these bound how much an omitted variable could explain)
# 2. Print the sensitivity summary
# 3. Plot the sensitivity contour using dml_plr.sensitivity_plot()
# 4. Interpret: What is the robustness value? If it is above 1.0,
#    what does that tell us about the credibility of the estimate?
#
# Context: We estimated that 401(k) eligibility increases net
# financial assets by approximately $[YOUR_ATE_HERE]. The concern
# is that an unobserved variable (e.g., financial literacy) could
# confound both eligibility and savings. The sensitivity analysis
# asks: how strong would this confounder need to be to make our
# estimate disappear?
#
# Do NOT rewrite my entire notebook - just provide the sensitivity
# analysis code and interpretation."""

---

## Digital Portfolio: P.R.I.M.E. README Prompt

Copy and paste the prompt below into Claude or ChatGPT to generate a professional
README for your GitHub repository. **Do NOT ask the AI to write Python code - only documentation.**

```
I need help writing a project description for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Demonstrated regularization bias: LASSO shrinks the treatment coefficient toward
  zero when applied naively to a causal estimation problem (simulated DGP, TRUE_ATE=5.0)
* Set up a DoubleML Partially Linear Regression (PLR) model using Random Forest
  nuisance learners with 5-fold cross-fitting on 401(k) pension plan data
* Estimated the Average Treatment Effect of 401(k) eligibility on net financial assets
* Conducted Conditional ATE (CATE) analysis by income quartile to detect treatment
  effect heterogeneity
* Visualized CATE estimates with confidence intervals across income subgroups
* Key finding: [FILL IN - what was the ATE? How did it vary by income quartile?]

**Please write a README.md entry including:**
1. Project Title: Causal ML - Double Machine Learning for 401(k) Policy Evaluation
2. Objective: A professional one-sentence summary
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Make this sound like a professional tech economist wrote it.
```

### Push to GitHub

```bash
cd econ-lab-24-double-ml
git add notebooks/ figures/ README.md
git commit -m "Lab 24: Causal ML - Double Machine Learning for 401(k) Policy"
git push origin main
```

Submit your GitHub repo link on Canvas.